In [ ]:
!pip install import-ipynb

In [ ]:
import sys
import os
from torch import nn
from image_encoder import ImageEncoder
from text_encoder import TextEncoder
from google.colab import drive
drive.mount('DRIVE_PATH')
os.chdir('CLIP_MODEL_PATH')

import import_ipynb
from config import define_image_config, define_text_config, define_proj_config

txt_config = define_text_config()
img_config = define_image_config()
proj_config = define_proj_config()

class LinearProjection(nn.Module):
    def __init__(self, input_dim=768, output_dim=512, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.linear(x))

class CLIP(nn.Module):
    def __init__(self):
        super().__init__()
        self.image_encoder = ImageEncoder(
            num_layers = img_config["num_layers"],
            input_dim = img_config["input_dim"],
            output_dim = img_config["output_dim"],
            patch_size = img_config["patch_size"],
            img_size = img_config["img_size"],
            num_heads = img_config["num_heads"],
            mlp_hidden_dim = img_config["mlp_hidden_dim"],
            dropout = img_config["dropout"]
        )
        self.text_encoder = TextEncoder(
            vocab_size = txt_config["vocab_size"],
            dim = txt_config["input_dim"],
            num_heads = txt_config["num_heads"],
            hidden_dim = txt_config["mlp_hidden_dim"],
            num_layers = txt_config["num_layers"],
            max_seq_len = txt_config["max_seq_len"],
            dropout = txt_config["dropout"]
          )

        # Projection layers
        self.image_projection = LinearProjection(
            input_dim = proj_config["input_dim"],
            output_dim = proj_config["final_out"],
            dropout = proj_config["dropout"]
          )
        self.text_projection = LinearProjection(
            input_dim = proj_config["input_dim"],
            output_dim = proj_config["final_out"],
            dropout = proj_config["dropout"]
          )

    def encode_image(self, image):
        img_features = self.image_projection(self.image_encoder(image))
        return img_features / img_features.norm(dim=-1, keepdim=True)

    def encode_text(self, text, mask=None):
        txt_features = self.text_projection(self.text_encoder(text, mask))
        return txt_features / txt_features.norm(dim=-1, keepdim=True)

    def forward(self, image, text, mask=None):
        img_features = self.encode_image(image)
        txt_features = self.encode_text(text, mask)

        return img_features, txt_features